# 01 — Clean Corpus

Reads raw JSONL dumps and produces clean, canonical JSONL files used by all downstream notebooks.

**Inputs (per subreddit, concatenated):**
- `gradadmissions`:
  - `r_gradadmissions_posts.jsonl` (repo root, Aug 2023–Jul 2025)
  - `r_gradadmissions_comments.jsonl` (repo root, Aug 2023–Jul 2025)
  - `data/raw/r_gradadmissions_2022_posts.jsonl` (Aug 2022–Jul 2023)
  - `data/raw/r_gradadmissions_2022_comments.jsonl` (Aug 2022–Jul 2023)
- `mscs`:
  - `data/raw/r_MSCS_posts.jsonl` (Aug 2023–Jul 2025)
  - `data/raw/r_MSCS_comments.jsonl` (Aug 2023–Jul 2025)
  - `data/raw/r_MSCS_2022_posts.jsonl` (Aug 2022–Jul 2023)
  - `data/raw/r_MSCS_2022_comments.jsonl` (Aug 2022–Jul 2023)

**Outputs:**
- `data/processed_v2/{SUBREDDIT}/posts_clean.jsonl` — one post per line: `id, author, created_dt, clean_text, score, num_comments`
- `data/processed_v2/{SUBREDDIT}/comments_clean.jsonl` — one comment per line: `id, author, created_dt, post_id, clean_text, score`

**Cleaning steps applied:**
1. Date/author validation — parse `created_utc`, drop deleted/removed/null authors and bodies
2. Dedup & bot filtering — deduplicate on `id`, drop `AutoModerator` and bot accounts
3. Text normalization — lowercase, strip URLs, strip non-alpha, collapse whitespace → `clean_text`
4. Comment→post mapping — derive `post_id` from `link_id` (strip `t3_` prefix)

In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# ── CONFIG ──────────────────────────────────────────────────────────────────
SUBREDDIT = 'gradadmissions'   # change to 'mscs' for MSCS pipeline
# ────────────────────────────────────────────────────────────────────────────

ROOT    = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
DATA_V2 = ROOT / 'data' / 'processed_v2' / SUBREDDIT
DATA_V2.mkdir(parents=True, exist_ok=True)

# Raw input paths per subreddit. Lists are concatenated (in order) before
# cleaning, so multiple admission cycles can be ingested together.
#   - gradadmissions: Aug 2023–Jul 2025 at repo root + Aug 2022–Jul 2023 in data/raw/
#   - mscs:           same pattern, all files live in data/raw/
_POSTS_IN = {
    'gradadmissions': [
        ROOT    / 'r_gradadmissions_posts.jsonl',
        RAW_DIR / 'r_gradadmissions_2022_posts.jsonl',
    ],
    'mscs': [
        RAW_DIR / 'r_MSCS_posts.jsonl',
        RAW_DIR / 'r_MSCS_2022_posts.jsonl',
    ],
}
_COMMENTS_IN = {
    'gradadmissions': [
        ROOT    / 'r_gradadmissions_comments.jsonl',
        RAW_DIR / 'r_gradadmissions_2022_comments.jsonl',
    ],
    'mscs': [
        RAW_DIR / 'r_MSCS_comments.jsonl',
        RAW_DIR / 'r_MSCS_2022_comments.jsonl',
    ],
}

POSTS_INS    = _POSTS_IN[SUBREDDIT]
COMMENTS_INS = _COMMENTS_IN[SUBREDDIT]
POSTS_OUT    = DATA_V2 / 'posts_clean.jsonl'
COMMENTS_OUT = DATA_V2 / 'comments_clean.jsonl'

print(f'Subreddit:   r/{SUBREDDIT}')
print('Posts in:')
for p in POSTS_INS:
    print(f'  - {p}  (exists: {p.exists()})')
print('Comments in:')
for p in COMMENTS_INS:
    print(f'  - {p}  (exists: {p.exists()})')
print('Out dir:    ', DATA_V2)

## Helper functions

In [18]:
_URL_RE   = re.compile(r'https?://\S+|www\.\S+')
_NONALPHA = re.compile(r'[^a-zA-Z\s]')
_SPACES   = re.compile(r'\s+')
_BOT_RE   = re.compile(r'(?i)bot$')

DELETED_VALS = {'[deleted]', '[removed]', 'None', None}
BOT_NAMES    = {'AutoModerator'}


def parse_utc(val) -> str | None:
    """Return ISO datetime string (UTC) from Unix timestamp or ISO string."""
    if val is None:
        return None
    try:
        return datetime.fromtimestamp(float(val), tz=timezone.utc).isoformat()
    except (TypeError, ValueError):
        pass
    try:
        return datetime.fromisoformat(str(val)).isoformat()
    except ValueError:
        return None


def clean_text(raw: str) -> str:
    """Lowercase, strip URLs, strip non-alpha, collapse whitespace."""
    t = _URL_RE.sub(' ', raw.lower())
    t = _NONALPHA.sub(' ', t)
    return _SPACES.sub(' ', t).strip()


def is_bot(author: str) -> bool:
    return author in BOT_NAMES or bool(_BOT_RE.search(author))


def is_deleted(val) -> bool:
    return val in DELETED_VALS or (
        isinstance(val, str) and val.strip() in {'[deleted]', '[removed]', ''}
    )


print('Helpers defined.')

Helpers defined.


## 1) Clean posts

In [ ]:
counts_posts = {}

raw_posts = []
for p in POSTS_INS:
    if not p.exists():
        print(f'  SKIP (missing): {p}')
        continue
    n_before = len(raw_posts)
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line:
                raw_posts.append(json.loads(line))
    print(f'  Loaded {len(raw_posts) - n_before:>7,} from {p.name}')

counts_posts['loaded'] = len(raw_posts)
print(f"Loaded (total):               {counts_posts['loaded']:>7,}")

In [20]:
# Step 1a: date/author/body validation
valid_posts = []
for r in raw_posts:
    dt = parse_utc(r.get('created_utc'))
    if dt is None:
        continue
    author = r.get('author')
    if is_deleted(author):
        continue
    selftext = r.get('selftext', '') or ''
    title    = r.get('title', '')    or ''
    combined = (title + ' ' + selftext).strip()
    if is_deleted(selftext) or len(combined) <= 5:
        continue
    r['_dt']       = dt
    r['_combined'] = combined
    valid_posts.append(r)

counts_posts['after_date_author'] = len(valid_posts)
print(f"After date/author validation: {counts_posts['after_date_author']:>7,}")

After date/author validation:  17,229


In [21]:
# Step 1b: dedup on id (keep first occurrence)
seen_ids = set()
deduped  = []
for r in valid_posts:
    if r['id'] not in seen_ids:
        seen_ids.add(r['id'])
        deduped.append(r)

counts_posts['after_dedup'] = len(deduped)
print(f"After dedup:                  {counts_posts['after_dedup']:>7,}")

After dedup:                   17,229


In [22]:
# Step 1c: bot filtering
no_bots = [r for r in deduped if not is_bot(str(r.get('author', '')))]

counts_posts['after_bot_filter'] = len(no_bots)
print(f"After bot filter:             {counts_posts['after_bot_filter']:>7,}")

After bot filter:              17,200


In [23]:
# Step 1d: text normalization → build output records
clean_posts = []
for r in no_bots:
    ct = clean_text(r['_combined'])
    if len(ct) < 5:
        continue
    clean_posts.append({
        'id':           r['id'],
        'author':       r['author'],
        'created_dt':   r['_dt'],
        'clean_text':   ct,
        'score':        r.get('score', 0),
        'num_comments': r.get('num_comments', 0),
    })

counts_posts['final'] = len(clean_posts)
print(f"Final clean posts:            {counts_posts['final']:>7,}")

Final clean posts:             17,200


In [24]:
print('\n--- Posts cleaning summary ---')
for step, n in counts_posts.items():
    print(f'  {step:<28} {n:>7,}')

print('\nSample (5 rows):')
pd.DataFrame(clean_posts[:5])[['id', 'author', 'created_dt', 'score', 'num_comments', 'clean_text']] \
  .assign(clean_text=lambda d: d['clean_text'].str[:80])


--- Posts cleaning summary ---
  loaded                        18,799
  after_date_author             17,229
  after_dedup                   17,229
  after_bot_filter              17,200
  final                         17,200

Sample (5 rows):


,id,author,created_dt,score,num_comments,clean_text
0,15f3g0b,08Satan,2023-08-01T05:41:51+00:00,2,16,ec undergraduate here is it nearly impossible ...
1,15ffeap,han_1206,2023-08-01T15:14:17+00:00,3,18,profile evaluation for fall hey fellow reddito...
2,15fgn9t,igar234,2023-08-01T16:00:50+00:00,1,2,what transcripts to send to universities durin...
3,15g7mj1,dopeandcope,2023-08-02T12:27:47+00:00,6,2,ms cs vs meng cs vs ms in software engineering...
4,15g8yv2,Sad_Length4576,2023-08-02T13:29:11+00:00,4,6,how to find out if a college offers a large nu...


## 2) Clean comments

In [ ]:
counts_comments = {}

raw_comments = []
for p in COMMENTS_INS:
    if not p.exists():
        print(f'  SKIP (missing): {p}')
        continue
    n_before = len(raw_comments)
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line:
                raw_comments.append(json.loads(line))
    print(f'  Loaded {len(raw_comments) - n_before:>7,} from {p.name}')

counts_comments['loaded'] = len(raw_comments)
print(f"Loaded (total):               {counts_comments['loaded']:>7,}")

In [26]:
# Step 2a: date/author/body validation
valid_comments = []
for r in raw_comments:
    dt = parse_utc(r.get('created_utc'))
    if dt is None:
        continue
    author = r.get('author')
    if is_deleted(author):
        continue
    body = r.get('body', '') or ''
    if is_deleted(body) or len(body.strip()) <= 5:
        continue
    link_id = r.get('link_id', '')
    if not link_id:
        continue
    r['_dt'] = dt
    valid_comments.append(r)

counts_comments['after_date_author'] = len(valid_comments)
print(f"After date/author validation: {counts_comments['after_date_author']:>7,}")

After date/author validation: 125,772


In [27]:
# Step 2b: dedup on id
seen_ids = set()
deduped  = []
for r in valid_comments:
    if r['id'] not in seen_ids:
        seen_ids.add(r['id'])
        deduped.append(r)

counts_comments['after_dedup'] = len(deduped)
print(f"After dedup:                  {counts_comments['after_dedup']:>7,}")

After dedup:                  125,772


In [28]:
# Step 2c: bot filtering
no_bots = [r for r in deduped if not is_bot(str(r.get('author', '')))]

counts_comments['after_bot_filter'] = len(no_bots)
print(f"After bot filter:             {counts_comments['after_bot_filter']:>7,}")

After bot filter:             125,512


In [29]:
# Step 2d: text normalization + comment→post mapping
clean_comments = []
for r in no_bots:
    ct = clean_text(r['body'])
    if len(ct) < 5:
        continue
    link_id = r.get('link_id', '')
    post_id = link_id.removeprefix('t3_') if link_id else ''
    clean_comments.append({
        'id':         r['id'],
        'author':     r['author'],
        'created_dt': r['_dt'],
        'post_id':    post_id,
        'clean_text': ct,
        'score':      r.get('score', 0),
    })

counts_comments['final'] = len(clean_comments)
print(f"Final clean comments:         {counts_comments['final']:>7,}")

Final clean comments:         124,547


In [30]:
print('\n--- Comments cleaning summary ---')
for step, n in counts_comments.items():
    print(f'  {step:<28} {n:>7,}')

print('\nSample (5 rows) — check post_id populated:')
pd.DataFrame(clean_comments[:5])[['id', 'author', 'created_dt', 'post_id', 'score', 'clean_text']] \
  .assign(clean_text=lambda d: d['clean_text'].str[:60])


--- Comments cleaning summary ---
  loaded                       133,482
  after_date_author            125,772
  after_dedup                  125,772
  after_bot_filter             125,512
  final                        124,547

Sample (5 rows) — check post_id populated:


,id,author,created_dt,post_id,score,clean_text
0,jua0ois,_PandaBear,2023-08-01T00:17:26+00:00,15edup5,5,i would always go for us i know it has things ...
1,juadbmb,fall2023mscs,2023-08-01T01:50:36+00:00,15ea1qx,2,doesn t matter did it in undergrad now going f...
2,juaiyg0,admit_2024,2023-08-01T02:33:56+00:00,15eczio,2,yeah reading posts about last cycle being brut...
3,juast2d,chanchanmano,2023-08-01T03:58:12+00:00,15ej7x7,1,i don t hope not if you have a great cgpa a fe...
4,jub0t8t,TheEvilHBK,2023-08-01T05:18:56+00:00,15ea1qx,1,people have completely misunderstood what i wa...


In [31]:
# Verify post_id→post cross-reference
post_ids_set      = {p['id'] for p in clean_posts}
comment_post_ids  = {c['post_id'] for c in clean_comments}
overlap           = len(comment_post_ids & post_ids_set)
print(f"Unique post_ids in comments:   {len(comment_post_ids):>6,}")
print(f"Of those found in posts_clean: {overlap:>6,}  ({100*overlap/max(len(comment_post_ids),1):.1f}%)")

Unique post_ids in comments:   14,014
Of those found in posts_clean: 13,112  (93.6%)


## 3) Write output files

> **Approval gate:** Review the summaries and samples above before running this cell.

In [32]:
with open(POSTS_OUT, 'w') as f:
    for rec in clean_posts:
        f.write(json.dumps(rec) + '\n')
print(f'Wrote {len(clean_posts):,} posts → {POSTS_OUT}')

with open(COMMENTS_OUT, 'w') as f:
    for rec in clean_comments:
        f.write(json.dumps(rec) + '\n')
print(f'Wrote {len(clean_comments):,} comments → {COMMENTS_OUT}')

Wrote 17,200 posts → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/mscs/posts_clean.jsonl
Wrote 124,547 comments → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/mscs/comments_clean.jsonl
